In [24]:
import torch.nn as nn
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import pandas as pd
from collections import Counter
import random
import ast

In [25]:
df = pd.read_csv("num_dataset_10.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [26]:
go_val = 10       #fed as the decoder's very first input only; never appears in the data itself
pad_val = 11
vocab_size = 12

In [27]:
inp_tensors = [torch.tensor(ast.literal_eval(s), dtype=torch.long) for s in df['source']]
out_tensors = [torch.tensor(ast.literal_eval(s), dtype=torch.long) for s in df['target']]

In [28]:
inp_padded = nn.utils.rnn.pad_sequence(inp_tensors, batch_first=True, padding_value=pad_val)
out_padded = nn.utils.rnn.pad_sequence(out_tensors, batch_first=True, padding_value=pad_val)

In [29]:
dataset = TensorDataset(inp_padded, out_padded)

#80, 10, 10 split for Train, Val, Test
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset,  batch_size=128, shuffle=False)

In [30]:
class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size, num_layers):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        #Expands a token to dim embed_size, based on learned embeddings
        embedding = self.dropout(self.embedding(x))

        #outputs: hidden state at every timestep, shape (batch, seq_len, hidden_size)
        #         (not used by the decoder - no attention here)
        #hidden:  final hidden state, shape (num_layers, batch, hidden_size)
        #         used to initialize the decoder's hidden state
        outputs, hidden = self.gru(embedding)

        return outputs, hidden

In [31]:
class Decoder(nn.Module):
    #Input and output sizes are the same here b/c both use same vocab space
    def __init__(self, input_size, embed_size, hidden_size, output_size, num_layers):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, embed_size)

        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.2)

        #fc takes the GRU output straight to vocab_size
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        #Converts shape of x from (batch_size) to (batch_size, 1)
        x = x.unsqueeze(1)
        #Converts shape now to (batch_size, 1, embed_size)
        embedding = self.dropout(self.embedding(x))

        outputs, hidden = self.gru(embedding, hidden)

        #Squeeze the seq_len=1 dimension back out
        outputs = outputs.squeeze(1)

        predictions = self.fc(outputs)

        return predictions, hidden

In [32]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target, teacher_force_ratio):
        batch_size = source.shape[0]
        target_len = source.shape[1]

        #Tensor of zeros with shape (batch_size, target_len, vocab_size), will gradually be filled in by the decoder
        #BN: Vocab_size != Embedding_Size
        outputs = torch.zeros(batch_size, target_len, vocab_size).to(device)

        #Only the final hidden state is used to seed the decoder now
        _, hidden = self.encoder(source)

        #<go> is not part of the source/target data - it only seeds the decoder's first input
        x = torch.full((batch_size,), go_val, dtype=torch.long).to(device)

        for t in range(target_len):
            output, hidden = self.decoder(x, hidden)
            #Fill in the outputs variable
            outputs[:, t] = output

            #Chooses the one with the highest probability
            best_guess = output.argmax(1)

            #Using random to ensure teacher forcing is applied only a certain percentage of the time
            if random.random() < teacher_force_ratio:
                x = target[:, t]
            else:
                x = best_guess

        return outputs

In [33]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, loader, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for src_batch, trg_batch in loader:
            src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
            output = model(src_batch, trg_batch, 0.0)
            output = output.reshape(-1, vocab_size)
            target = trg_batch.reshape(-1)
            loss = criterion(output, target)
            epoch_loss += loss.item()

        return epoch_loss / len(loader)

In [34]:
#Find the models accuracy on a Dataloader
def accuracy(model, loader, device):
    with torch.no_grad():
        model.eval()
        accurate = 0
        total = 0

        for src_batch, trg_batch in loader:
            src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
            output = model(src_batch, trg_batch, 0.0)

            output = output.reshape(-1, vocab_size)
            target = trg_batch.reshape(-1)

            #Creates list of booleans, False if 0, else True
            mask = target != pad_val
            preds = output.argmax(dim=1)

            #Ignoring padding tokens when evaluating
            accurate += (preds[mask] == target[mask]).sum().item()
            total += mask.sum().item()

        return accurate/total

In [35]:
def test(model, test_seq):
    with torch.no_grad():
        model.eval()
        encoded = torch.tensor(test_seq, dtype=torch.long)

        #Pad our text to the length the model was trained on
        pad_len = inp_padded.shape[1] - len(encoded)
        encoded = torch.cat([encoded, torch.zeros(pad_len, dtype=torch.long).fill_(pad_val)])

        #Gives it batch size 1
        encoded = encoded.unsqueeze(0).to(device)

        #Final hidden state seeds the decoder
        _, hidden = model.encoder(encoded)

        predicted = []
        #Seed the decoder with the <go> token - it's never part of the sequence itself
        decoder_input = torch.tensor([go_val], device=device)

        #No <eos> to signal a stop, so we rely on knowing the output length:
        #this is a copy task, so the output is exactly as long as the input
        for step in range(len(test_seq)):
            prediction, hidden = model.decoder(decoder_input, hidden)
            best_guess = prediction.argmax(1).item()

            predicted.append(best_guess)
            decoder_input = torch.tensor([best_guess], device=device)

        return predicted

In [36]:
#Training hyperparameters
num_epochs = 50
learning_rate = 1e-3
encoder_embedding_size = 64
decoder_embedding_size = 64
hidden_size = 128
num_layers = 1

#Model hyperparameters
encoder_net = Encoder(vocab_size, encoder_embedding_size, hidden_size, num_layers).to(device)
decoder_net = Decoder(vocab_size, decoder_embedding_size, hidden_size, vocab_size, num_layers).to(device)

model = Seq2Seq(encoder_net, decoder_net).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
#Ignore <pad>
criterion = nn.CrossEntropyLoss(ignore_index=pad_val)
#Combat overfitting
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7
)

In [37]:
test_str = [8, 0, 3, 6, 5]
tf_ratio = 1
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
      for src_batch, trg_batch in train_loader:
          src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
          optimizer.zero_grad()

          output = model(src_batch, trg_batch, max(0.0, tf_ratio))

          #Flatten the batch size and seq length dimensions into 1
          output = output.reshape(-1, vocab_size)
          target = trg_batch.reshape(-1)

          loss = criterion(output, target)
          loss.backward()

          #Norm=magnitude, clipping gradient to ensure they don't explode
          torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1)
          optimizer.step()
          epoch_loss += loss.item()

    print(f"{test_str} => {test(model, test_str)}")
    train_loss = epoch_loss/len(train_loader)
    print(f"Train Loss: {train_loss}")
    val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Validation Loss: {val_loss}")
    tf_ratio -= 0.025
    scheduler.step(val_loss)

Epoch 1 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.7529759944081307
Validation Loss: 0.49736766966562423
Epoch 2 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.2128481304347515
Validation Loss: 0.24867509825835152
Epoch 3 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.13603889134526254
Validation Loss: 0.15596352913786496
Epoch 4 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.09903121744841337
Validation Loss: 0.15350824127358104
Epoch 5 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.08255396945774555
Validation Loss: 0.08935108884341186
Epoch 6 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.0594041066840291
Validation Loss: 0.08308573212060663
Epoch 7 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.05332492349669337
Validation Loss: 0.0915749113829363
Epoch 8 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.04138031723350286
Validation Loss: 0.04305034386555827
Epoch 9 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.036398

In [38]:
print(f"Train Loss: {evaluate(model, train_loader, criterion, device)}")
print(f"Train Acc: {accuracy(model, train_loader, device)}")
print(f"Val Loss: {evaluate(model, val_loader, criterion, device)}")
print(f"Val Acc: {accuracy(model, val_loader, device)}")
print(f"Test Loss: {evaluate(model, test_loader, criterion, device)}")
print(f"Test Acc: {accuracy(model, test_loader, device)}")

Train Loss: 0.0015349635733291507
Train Acc: 0.9999950017618789
Val Loss: 0.001606665344701873
Val Acc: 1.0
Test Loss: 0.0017443895081265105
Test Acc: 0.9999599447237187


In [39]:
torch.save(model.state_dict(), "no_sos_eos_gru_num_reversal_model.pt")
# model.load_state_dict(torch.load("gru_num_copy_model.pt", weights_only=True, map_location=torch.device('cuda')))

In [40]:
#The encoder's final hidden state is the actual context vector the decoder is
#seeded with (there's no attention here, so this is exactly what gets passed along).
def build_context_df(model, loader, inp_padded, device):
    model.eval()
    all_sources = []
    all_targets = []
    all_contexts = []

    with torch.no_grad():
        for src_batch, trg_batch in loader:
            src_batch = src_batch.to(device)
            trg_batch = trg_batch.to(device)
            _, hidden = model.encoder(src_batch)

            context = hidden[0]

            all_sources.append(src_batch.cpu())
            all_targets.append(trg_batch.cpu())
            all_contexts.append(context.cpu())

    all_sources = torch.cat(all_sources, dim=0)   # (N, seq_len)
    all_targets = torch.cat(all_targets, dim=0)   # (N, seq_len)
    all_contexts = torch.cat(all_contexts, dim=0)  # (N, hidden_size)

    source_lists = [row.tolist() for row in all_sources]
    target_lists = [row.tolist() for row in all_targets]
    context_lists = [row.tolist() for row in all_contexts]

    context_df = pd.DataFrame({
        "source": source_lists,
        "context_vector": context_lists,
        "target": target_lists
    })

    return context_df

full_loader = DataLoader(dataset, batch_size=32, shuffle=False)
context_df = build_context_df(model, full_loader, inp_padded, device)

In [41]:
context_df.head()

,source,context_vector,target
0,"[8, 1, 8, 2, 1, 6, 11, 11, 11]","[0.09410955756902695, -0.8832139372825623, 0.9...","[6, 1, 2, 8, 1, 8, 11, 11, 11]"
1,"[2, 3, 3, 7, 9, 6, 11, 11, 11]","[0.052860140800476074, -0.5572207570075989, 0....","[6, 9, 7, 3, 3, 2, 11, 11, 11]"
2,"[8, 5, 9, 4, 6, 7, 11, 11, 11]","[0.02710636518895626, -0.8334079384803772, 0.8...","[7, 6, 4, 9, 5, 8, 11, 11, 11]"
3,"[0, 6, 5, 9, 8, 7, 11, 11, 11]","[-0.027502819895744324, -0.6997084021568298, 0...","[7, 8, 9, 5, 6, 0, 11, 11, 11]"
4,"[2, 7, 3, 4, 4, 11, 11, 11, 11]","[-0.027194328606128693, -0.8425125479698181, 0...","[4, 4, 3, 7, 2, 11, 11, 11, 11]"


In [42]:
def strip_padding(tokens):
    while tokens and tokens[-1] == pad_val:
        tokens = tokens[:-1]
    return tokens

context_df["source"] = context_df["source"].apply(strip_padding)
context_df["target"] = context_df["target"].apply(strip_padding)

In [43]:
display(context_df)

,source,context_vector,target
0,"[8, 1, 8, 2, 1, 6]","[0.09410955756902695, -0.8832139372825623, 0.9...","[6, 1, 2, 8, 1, 8]"
1,"[2, 3, 3, 7, 9, 6]","[0.052860140800476074, -0.5572207570075989, 0....","[6, 9, 7, 3, 3, 2]"
2,"[8, 5, 9, 4, 6, 7]","[0.02710636518895626, -0.8334079384803772, 0.8...","[7, 6, 4, 9, 5, 8]"
3,"[0, 6, 5, 9, 8, 7]","[-0.027502819895744324, -0.6997084021568298, 0...","[7, 8, 9, 5, 6, 0]"
4,"[2, 7, 3, 4, 4]","[-0.027194328606128693, -0.8425125479698181, 0...","[4, 4, 3, 7, 2]"
...,...,...,...
79989,"[4, 7, 0, 6, 4, 3, 7, 2]","[-0.4898410737514496, -0.9093802571296692, 0.9...","[2, 7, 3, 4, 6, 0, 7, 4]"
79990,"[2, 3, 5, 3, 5, 0, 5, 1, 9]","[0.8303380012512207, 0.6530735492706299, 0.390...","[9, 1, 5, 0, 5, 3, 5, 3, 2]"
79991,"[2, 1, 7, 7, 9, 6, 5]","[0.25535833835601807, -0.8208686113357544, -0....","[5, 6, 9, 7, 7, 1, 2]"
79992,"[3, 1, 8, 2, 4, 6, 2, 8]","[0.37571385502815247, -0.9174439311027527, 0.9...","[8, 2, 6, 4, 2, 8, 1, 3]"


In [44]:
context_df.to_csv("no_sos_eos_gru_reversal_data.csv")